In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
mandi_df = pd.read_csv(r'C:\Users\aliii\OneDrive\Desktop\Yeild2Profit\data\mandi_prices_real.csv')

In [ ]:
mandi_df.head()

In [ ]:
yield_df = pd.read_csv(r'C:\Users\aliii\OneDrive\Desktop\Yeild2Profit\data\crop_yield_real.csv')

In [ ]:
yield_df.head()

In [ ]:
mandi_df.info()

In [ ]:
yield_df.info()

In [ ]:
mandi_df.isnull().sum()

In [ ]:
yield_df.isnull().sum()

In [ ]:
mandi_df.columns = [c.strip().lower().replace(' ', '_') for c in mandi_df.columns]
mandi_df.rename(columns={'commodity': 'crop', 'modal_price': 'modal_price_per_quintal'}, inplace=True)
mandi_df['crop']  = mandi_df['crop'].str.strip().str.title()
mandi_df['state'] = mandi_df['state'].str.strip().str.title()
mandi_df.dropna(inplace=True)
mandi_df = mandi_df[mandi_df['modal_price_per_quintal'] > 0]
mandi_df.head()

In [ ]:
yield_df.columns = [c.strip().lower().replace(' ', '_') for c in yield_df.columns]
yield_df.rename(columns={'yield': 'yield_per_hectare'}, inplace=True)
yield_df['crop']   = yield_df['crop'].str.strip().str.title()
yield_df['state']  = yield_df['state'].str.strip().str.title()
yield_df['season'] = yield_df['season'].str.strip().str.title()
yield_df.dropna(inplace=True)
yield_df = yield_df[yield_df['yield_per_hectare'] > 0]
yield_df.head()

In [ ]:
dup = mandi_df[mandi_df.duplicated()]
mandi_df = mandi_df.drop(dup.index, axis=0)
print('Duplicates removed:', len(dup))

In [ ]:
market_avg = mandi_df.groupby(['crop', 'state'])['modal_price_per_quintal'].mean().reset_index()
market_avg['modal_price_per_quintal'] = market_avg['modal_price_per_quintal'].round(2)
market_avg.head()

In [ ]:
yield_avg = yield_df.groupby(['crop', 'state', 'season'])['yield_per_hectare'].mean().reset_index()
yield_avg['yield_per_hectare'] = yield_avg['yield_per_hectare'].round(4)
yield_avg.head()

In [ ]:
df = yield_avg.merge(market_avg, on=['crop', 'state'], how='inner')
df.head()

In [ ]:
df.shape

In [ ]:
cost_map = {
    'Rice': 35000, 'Wheat': 30000, 'Maize': 22000,
    'Jowar': 15000, 'Bajra': 13000, 'Ragi': 16000,
    'Arhar/Tur': 18000, 'Moong(Green Gram)': 17000,
    'Urad': 16000, 'Gram': 16000, 'Lentil': 15000,
    'Groundnut': 28000, 'Soyabean': 20000,
    'Sunflower': 18000, 'Rapeseed &Mustard': 14000,
    'Sesamum': 14000, 'Cotton(Lint)': 38000,
    'Sugarcane': 80000, 'Jute': 30000,
    'Onion': 50000, 'Potato': 55000, 'Tomato': 60000,
    'Ginger': 90000, 'Turmeric': 70000, 'Banana': 120000,
    'Coconut': 40000, 'Coffee': 85000, 'Tea': 100000,
    'Castor Seed': 18000, 'Linseed': 14000, 'Arecanut': 80000,
}

df['cost_per_hectare'] = df['crop'].map(cost_map)
mask = df['cost_per_hectare'].isna()
df.loc[mask, 'cost_per_hectare'] = (
    df.loc[mask, 'yield_per_hectare'] * 10 *
    df.loc[mask, 'modal_price_per_quintal'] * 0.55
).round(2)

df['revenue_per_hectare'] = (df['yield_per_hectare'] * 10 * df['modal_price_per_quintal']).round(2)
df['profit_per_hectare']  = (df['revenue_per_hectare'] - df['cost_per_hectare']).round(2)
df.dropna(inplace=True)
df.head()

In [ ]:
df.describe()

In [ ]:
q1  = df['profit_per_hectare'].quantile(0.25)
q3  = df['profit_per_hectare'].quantile(0.75)
iqr = q3 - q1
outliers = df[(df['profit_per_hectare'] < q1 - 1.5*iqr) | (df['profit_per_hectare'] > q3 + 1.5*iqr)]
outliers

In [ ]:
df = df.drop(outliers.index, axis=0)
df.shape

In [ ]:
plt.boxplot(df['profit_per_hectare'], vert=False)
plt.title('Profit per Hectare after Outlier Removal')
plt.xlabel('Profit (Rs)')
plt.show()

In [ ]:
top_crops = df.groupby('crop')['profit_per_hectare'].mean().sort_values(ascending=False).head(10)
top_crops.plot(kind='bar', color='green', edgecolor='black')
plt.title('Top 10 Crops by Average Profit per Hectare')
plt.xlabel('Crop')
plt.ylabel('Profit (Rs)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('season')['profit_per_hectare'].mean().plot(kind='bar', color='darkgreen', edgecolor='black')
plt.title('Average Profit by Season')
plt.xlabel('Season')
plt.ylabel('Profit (Rs)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
sns.heatmap(df[['yield_per_hectare','modal_price_per_quintal','cost_per_hectare','profit_per_hectare']].corr(),
            annot=True, cmap='Greens')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
np.random.seed(42)
rows = []
for _, ref in df.iterrows():
    for _ in range(100):
        area    = round(np.random.uniform(0.5, 5.0), 2)
        price_v = ref['modal_price_per_quintal'] * np.random.uniform(0.90, 1.10)
        yield_v = ref['yield_per_hectare']       * np.random.uniform(0.90, 1.10)
        cost_v  = ref['cost_per_hectare']        * np.random.uniform(0.95, 1.05)
        revenue = yield_v * 10 * price_v * area
        profit  = revenue - (cost_v * area)
        rows.append({'crop': ref['crop'], 'season': ref['season'], 'area': area,
                     'yield_per_hectare': round(yield_v,4), 'price_per_quintal': round(price_v,2),
                     'cost_per_hectare': round(cost_v,2), 'profit': round(profit,2)})

train_df = pd.DataFrame(rows)
train_df.head()

In [ ]:
train_df.shape

In [ ]:
le_crop   = LabelEncoder()
le_season = LabelEncoder()
train_df['crop_enc']   = le_crop.fit_transform(train_df['crop'])
train_df['season_enc'] = le_season.fit_transform(train_df['season'])
train_df.head()

In [ ]:
X = train_df[['crop_enc', 'season_enc', 'area', 'cost_per_hectare', 'price_per_quintal', 'yield_per_hectare']]
y = train_df['profit']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# Polynomial Regression
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly  = poly.transform(X_test_scaled)

model = LinearRegression()
model.fit(X_train_poly, y_train)

In [ ]:
y_pred = model.predict(X_test_poly)

In [ ]:
print('R2 Score:', r2_score(y_test, y_pred))

In [ ]:
print('MAE :', mean_absolute_error(y_test, y_pred))
print('MSE :', mean_squared_error(y_test, y_pred))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
# Linear Regression for comparison
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
print('Linear Regression R2    :', r2_score(y_test, y_pred_lr))
print('Polynomial Regression R2:', r2_score(y_test, y_pred))

In [ ]:
# Actual vs Predicted line graph
plt.figure(figsize=(12, 6))
plt.plot(y_test.values[:50], label='Actual Profit', color='blue', linewidth=2)
plt.plot(y_pred[:50], label='Predicted Profit', color='red', linestyle='--', linewidth=2)
plt.title('Yield2Profit: Actual vs Predicted Profit')
plt.xlabel('Sample Index')
plt.ylabel('Profit (Rs)')
plt.legend()
plt.show()

In [ ]:
plt.scatter(y_test, y_pred, color='green', alpha=0.5)
plt.xlabel('Actual Profit')
plt.ylabel('Predicted Profit')
plt.title('Actual vs Predicted')
plt.show()

In [ ]:
# Sample prediction - Wheat, Rabi, 2 hectares, Rs 60000 budget
ref = df[(df['crop'] == 'Wheat') & (df['season'] == 'Rabi')]

price     = ref['modal_price_per_quintal'].mean()
yield_val = ref['yield_per_hectare'].mean()
area      = 2.0
budget    = 60000
cost_ph   = budget / area

crop_enc   = le_crop.transform(['Wheat'])[0]
season_enc = le_season.transform(['Rabi'])[0]

print('Crop encoding for Wheat  :', crop_enc)
print('Season encoding for Rabi :', season_enc)
print('Market Price             :', round(price, 2))
print('Yield per hectare        :', round(yield_val, 4))

sample = pd.DataFrame(
    [[crop_enc, season_enc, area, cost_ph, price, yield_val]],
    columns=['crop_enc', 'season_enc', 'area', 'cost_per_hectare', 'price_per_quintal', 'yield_per_hectare']
)

sample_scaled = scaler.transform(sample)
sample_poly   = poly.transform(sample_scaled)
prediction    = model.predict(sample_poly)

manual_revenue = yield_val * 10 * price * area
manual_profit  = manual_revenue - budget

print()
print('Manual Revenue  :', round(manual_revenue, 2))
print('Manual Profit   :', round(manual_profit, 2))
print(f'Predicted Profit: {prediction[0]:.2f}')